In [6]:
# import features
import pandas as pd
# acitvity entropy
df_entropy = pd.read_csv('../output/data_cleaned/activity_entropy_rates.csv')
print(df_entropy.shape)

# early warning scores (physiological)
df_EWS = pd.read_csv('../output/data_cleaned/Mian_warning_score.csv')
df_EWS.columns = ['patient_id', 'date', 'early_warning_score']


# sleep quality
df_sleep_quality = pd.read_csv('../output/sleep_score/sleep_quality_score_by_duration.csv')
df_sleep_quality = df_sleep_quality.drop(columns= ['sum', 'scaled_sleep_quality_mean','scaled_sleep_quality_sum']).copy()
df_sleep_quality.columns = ['patient_id', 'date', 'sleep_quality_score']

# agitation
df_agitation = pd.read_csv('../output/data_cleaned/agitation_daily_counts.csv')

# uti
df_uti = pd.read_csv('../output/data_cleaned/uti_daily.csv')

merged_df = df_entropy
# merge dataframes
for df in [df_EWS, df_sleep_quality, df_agitation, df_uti]:
    print(df.shape)
    merged_df = pd.merge(merged_df, df, on=['patient_id', 'date'], how='outer')

# only consider the patients without NA in the following analysis (as the analysis itself will be individualized anyway)
# analysis_df = merged_df.dropna(subset=['sleep_quality_score']).dropna(subset=['early_warning_score']).dropna(subset=['entropy_rate']).copy()
analysis_df = merged_df.dropna(subset=['sleep_quality_score']).copy()
# fill uti and agitation none/NA as 0
cols = ['agitation_counts', 'uti_happen']
analysis_df[cols] = analysis_df[cols].fillna(0)
print(analysis_df.shape)
analysis_df

(2722, 3)
(2160, 3)
(800, 3)
(115, 3)
(265, 3)
(800, 7)


,patient_id,date,entropy_rate,early_warning_score,sleep_quality_score,agitation_counts,uti_happen
222,0f352,2019-06-26,0.669008,0.0,0.142857,0.0,1.0
223,0f352,2019-06-27,0.653618,NaN,1.200000,0.0,0.0
224,0f352,2019-06-28,0.613697,0.0,0.416667,0.0,0.0
225,0f352,2019-06-29,0.615494,0.0,1.000000,0.0,1.0
226,0f352,2019-06-30,0.514768,0.0,0.000000,0.0,0.0
...,...,...,...,...,...,...,...
2798,f220c,2019-06-22,0.608841,0.0,0.000000,0.0,0.0
2800,f220c,2019-06-24,0.600492,NaN,0.000000,0.0,0.0
2801,f220c,2019-06-25,0.518683,NaN,1.400000,0.0,0.0
2802,f220c,2019-06-26,0.621802,NaN,0.500000,0.0,0.0


## for each participant:
(1) keep valid sliding windows (at least 5 days in a 10-day window)
(2) For windows meeting the 5/10-day validity criterion, isolated missing days were imputed using each participant's within-window median value per predictor



In [7]:
# ── parameters ──────────────────────────────────────────────────────────────
WINDOW_SIZE = 10
MIN_VALID_DAYS = 5
PREDICTORS = ['entropy_rate', 'early_warning_score',
              'sleep_quality_score', 'agitation_counts', 'uti_happen']

analysis_df['date'] = pd.to_datetime(analysis_df['date'])
df = analysis_df.sort_values(['patient_id', 'date']).reset_index(drop=True).copy()


# ── sliding window per participant ───────────────────────────────────────────
results = []

for pid, pdata in df.groupby('patient_id'):
    pdata = pdata.set_index('date').asfreq('D')  # fill missing dates with NaN rows
    pdata['patient_id'] = pid
    dates = pdata.index

    for i in range(len(dates) - WINDOW_SIZE):  # note: not WINDOW_SIZE + 1
        baseline = pdata.iloc[i:i + WINDOW_SIZE].copy()      # days 1-10
        test_day = pdata.iloc[i + WINDOW_SIZE].copy()         # day 11

        # validity check on baseline only
        valid = all(baseline[col].notna().sum() >= MIN_VALID_DAYS
                    for col in PREDICTORS)

        if not valid:
            continue  # discard this baseline-test pair

        # impute baseline missing days with within-window median
        for col in PREDICTORS:
            col_median = baseline[col].median()
            baseline[col] = baseline[col].fillna(col_median)

        # check test day is not missing
        if test_day[PREDICTORS].isna().any():
            continue

        # store both together
        baseline['window_start'] = dates[i]
        baseline['window_end'] = dates[i + WINDOW_SIZE - 1]
        baseline['test_date'] = dates[i + WINDOW_SIZE]
        baseline['is_test_day'] = False

        test_row = test_day.to_frame().T.copy()
        test_row['window_start'] = dates[i]
        test_row['window_end'] = dates[i + WINDOW_SIZE - 1]
        test_row['test_date'] = dates[i + WINDOW_SIZE]
        test_row['is_test_day'] = True

        results.append(pd.concat([baseline, test_row]))

# ── combine ──────────────────────────────────────────────────────────────────
windowed_df = pd.concat(results).reset_index(names='date')
windowed_df

,date,patient_id,entropy_rate,early_warning_score,sleep_quality_score,agitation_counts,uti_happen,window_start,window_end,test_date,is_test_day
0,2019-04-28,16f4b,0.651879,0.0,1.5,0.0,0.0,2019-04-28,2019-05-07,2019-05-08,False
1,2019-04-29,16f4b,0.704693,0.0,1.5,0.0,0.0,2019-04-28,2019-05-07,2019-05-08,False
2,2019-04-30,16f4b,0.60318,0.0,1.0,0.0,0.0,2019-04-28,2019-05-07,2019-05-08,False
3,2019-05-01,16f4b,0.651631,0.0,1.2,0.0,0.0,2019-04-28,2019-05-07,2019-05-08,False
4,2019-05-02,16f4b,0.668107,0.0,1.0,0.0,0.0,2019-04-28,2019-05-07,2019-05-08,False
...,...,...,...,...,...,...,...,...,...,...,...
5407,2019-06-21,ec812,0.661363,0.0,1.125,0.0,0.0,2019-06-15,2019-06-24,2019-06-25,False
5408,2019-06-22,ec812,0.62356,0.0,0.666667,0.0,0.0,2019-06-15,2019-06-24,2019-06-25,False
5409,2019-06-23,ec812,0.708682,0.0,0.875,0.0,0.0,2019-06-15,2019-06-24,2019-06-25,False
5410,2019-06-24,ec812,0.65633,0.0,0.888889,0.0,0.0,2019-06-15,2019-06-24,2019-06-25,False


In [8]:
# total window number & check if imputation removes missingness
print(f"Total windows retained: {windowed_df.groupby(['patient_id','window_start']).ngroups}")
print(windowed_df.isnull().sum())  # should be zero after imputation

Total windows retained: 492
date                   0
patient_id             0
entropy_rate           0
early_warning_score    0
sleep_quality_score    0
agitation_counts       0
uti_happen             0
window_start           0
window_end             0
test_date              0
is_test_day            0
dtype: int64


In [9]:
windowed_df.to_csv('../output/windown_sliced_data/windowed_df_10d.csv', index=False)
windowed_df

,date,patient_id,entropy_rate,early_warning_score,sleep_quality_score,agitation_counts,uti_happen,window_start,window_end,test_date,is_test_day
0,2019-04-28,16f4b,0.651879,0.0,1.5,0.0,0.0,2019-04-28,2019-05-07,2019-05-08,False
1,2019-04-29,16f4b,0.704693,0.0,1.5,0.0,0.0,2019-04-28,2019-05-07,2019-05-08,False
2,2019-04-30,16f4b,0.60318,0.0,1.0,0.0,0.0,2019-04-28,2019-05-07,2019-05-08,False
3,2019-05-01,16f4b,0.651631,0.0,1.2,0.0,0.0,2019-04-28,2019-05-07,2019-05-08,False
4,2019-05-02,16f4b,0.668107,0.0,1.0,0.0,0.0,2019-04-28,2019-05-07,2019-05-08,False
...,...,...,...,...,...,...,...,...,...,...,...
5407,2019-06-21,ec812,0.661363,0.0,1.125,0.0,0.0,2019-06-15,2019-06-24,2019-06-25,False
5408,2019-06-22,ec812,0.62356,0.0,0.666667,0.0,0.0,2019-06-15,2019-06-24,2019-06-25,False
5409,2019-06-23,ec812,0.708682,0.0,0.875,0.0,0.0,2019-06-15,2019-06-24,2019-06-25,False
5410,2019-06-24,ec812,0.65633,0.0,0.888889,0.0,0.0,2019-06-15,2019-06-24,2019-06-25,False


In [10]:
# get the starting and ending dates for study
earliest_date = windowed_df["date"].min()
latest_date = windowed_df["date"].max()

print("Earliest date:", earliest_date)
print("Latest date:", latest_date)

Earliest date: 2019-04-01 00:00:00
Latest date: 2019-06-30 00:00:00


# missingness checking

In [11]:
# ── per-participant diagnostics ───────────────────────────────────────────────
rows = []

for pid, pdata in df.groupby("patient_id"):
    # --- missing-date % ---
    first_date = pdata["date"].min()
    last_date  = pdata["date"].max()
    span_days  = (last_date - first_date).days + 1          # inclusive
    recorded   = pdata["date"].nunique()
    missing_dates    = span_days - recorded
    missing_date_pct = 100 * missing_dates / span_days

    # --- discarded-window % (mirrors Cell 3) ---
    pdata_idx = pdata.set_index("date").asfreq("D")
    pdata_idx["patient_id"] = pid
    dates = pdata_idx.index

    total_windows     = 0
    discarded_windows = 0

    for i in range(len(dates) - WINDOW_SIZE):
        total_windows += 1
        baseline = pdata_idx.iloc[i : i + WINDOW_SIZE]
        test_day = pdata_idx.iloc[i + WINDOW_SIZE]

        valid = all(
            baseline[col].notna().sum() >= MIN_VALID_DAYS
            for col in PREDICTORS
        )
        if not valid:
            discarded_windows += 1
            continue

        if test_day[PREDICTORS].isna().any():
            discarded_windows += 1

    retained_windows = total_windows - discarded_windows
    discarded_pct = (
        100 * discarded_windows / total_windows if total_windows > 0 else float("nan")
    )
    retained_pct = 100 - discarded_pct if total_windows > 0 else float("nan")

    rows.append(
        {
            "patient_id"          : pid,
            # "first_date"          : first_date.date(),
            # "last_date"           : last_date.date(),
            "span_days"           : span_days,
            "recorded_days"       : recorded,
            "missing_days"        : missing_dates,
            "missing_date_pct"    : round(missing_date_pct, 1),
            "total_windows"       : total_windows,
            "discarded_windows"   : discarded_windows,
            "retained_windows"    : retained_windows,
            "discarded_window_pct": round(discarded_pct, 1),
            "retained_window_pct" : round(retained_pct, 1),
        }
    )

stats = pd.DataFrame(rows).sort_values("patient_id")

# ── split into included / excluded ───────────────────────────────────────────
included = stats[stats["retained_windows"] > 0].copy()
excluded = stats[stats["retained_windows"] == 0].copy()

# ── print ─────────────────────────────────────────────────────────────────────
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:.1f}".format)

COLS = [
    "patient_id",
    # "first_date", "last_date",
    "span_days",
    "recorded_days", "missing_days", "missing_date_pct",
    "total_windows", "discarded_windows", "retained_windows",
    "discarded_window_pct",
]

print("\n=== Participants with ≥ 1 retained window ===\n")


if not excluded.empty:
    print(f"\n=== Participants fully excluded (0 retained windows): n={len(excluded)} ===\n")
    print(excluded[["patient_id", "span_days", "recorded_days",
                    "total_windows", "discarded_windows"]].to_string(index=False))

print("\n=== Aggregate summary (included participants only) ===")
print(f"Included participants:      {len(included)} / {len(stats)}")
print(f"Median missing-date %:     {included['missing_date_pct'].median():.1f}%")
print(f"Range missing-date %:      {included['missing_date_pct'].min():.1f}% – {included['missing_date_pct'].max():.1f}%")
print(f"Median discarded-window %: {included['discarded_window_pct'].median():.1f}%")
print(f"Range discarded-window %:  {included['discarded_window_pct'].min():.1f}% – {included['discarded_window_pct'].max():.1f}%")



=== Participants with ≥ 1 retained window ===


=== Participants fully excluded (0 retained windows): n=4 ===

patient_id  span_days  recorded_days  total_windows  discarded_windows
     0f352          6              6              0                  0
     76230          4              4              0                  0
     b0455         10             10              0                  0
     f220c         64             45             54                 54

=== Aggregate summary (included participants only) ===
Included participants:      13 / 17
Median missing-date %:     2.6%
Range missing-date %:      0.0% – 31.6%
Median discarded-window %: 20.5%
Range discarded-window %:  3.4% – 97.1%


In [21]:
# get the person-days and median of recorded days across all participants

column_sum = included["recorded_days"].sum()
column_median = included["recorded_days"].median()
column_range = (included["recorded_days"].min() , included["recorded_days"].max())

print("Range:", column_range)
print("Sum:", column_sum)
print("Median:", column_median)

Range: (np.int64(18), np.int64(83))
Sum: 735
Median: 56.0
